In [108]:
!pip install -qU langchain

In [152]:
!pip install -qU langchain-google-genai langchain_groq


Model config


In [153]:
from langchain.chat_models import init_chat_model
from google.colab import userdata
from pprint import pprint

In [154]:
groq_api_key = userdata.get('groq_api_key')

In [155]:
model = init_chat_model(
    model = "openai/gpt-oss-120b",
    model_provider="groq",
    api_key = groq_api_key
)

Creating Skill demand Tool

In [90]:
!pip install -qU langchain-tavily

In [112]:
from langchain_tavily import TavilySearch

In [113]:
tavily_api_key = userdata.get('TAVILY_API_KEY')

In [126]:
skill_demand_tool = TavilySearch(
    max_results = 5,
    topic = "general",
    search_depth="advanced",
    tavily_api_key = tavily_api_key

)

Creating JSearch Tool

In [94]:
from langchain.tools import tool

In [95]:
rapid_api_key = userdata.get('RAPID_API_KEY')

In [127]:
@tool
def search_jobs(skill: str, location: str) -> list:
  """Search for jobs requiring specific skills using JSearch from RapidAPI."""
  print(f"\nCalling search_jobs tool")

  print(f"Skill: {skill}")
  print(f"Location: {location}")

  import requests
  url = "https://jsearch.p.rapidapi.com/search-v2"
  querystring = {"query":f" {skill} jobs in {location}",
                "country":"in",
                "employment_types":"FULLTIME,INTERN",
                "page":"1",
                "no_of_pages":"1",
                "job_requirements":"under_3_years_experience, no_experience"
                }
  headers = {
    "x-rapidapi-key": rapid_api_key,
    "x-rapidapi-host": "jsearch.p.rapidapi.com"
  }

  response = requests.get(url, headers=headers, params=querystring)
  data = response.json()

  # Correctly extract the jobs list from the nested dictionary
  jobs = data.get('data', {}).get('jobs', [])
  print(f"Found {len(jobs)} jobs")

  result = []
  for job in jobs:
    result.append({
        "title": job.get('job_title', ''),
        "company": job.get('employer_name', ''),
        "location": job.get('job_city', ''),
        "description": job.get('job_description', ''),
        "url": job.get('job_apply_link', '')
        })
  return result

Creating Agent

In [116]:
from langchain.agents import create_agent

In [128]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [156]:
agent = create_agent(
    model = model,
    tools = [skill_demand_tool, search_jobs],
    system_prompt=system_prompt,
    debug = True
)

In [119]:
user_query = "What's the demand for generative ai in the industry and show me related job openings in India"

In [157]:
response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})

[values] {'messages': [HumanMessage(content="What's the demand for generative ai in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='1045301d-40b8-43bf-b5d6-c1808f0d51f3')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide demand info and job openings in India for generative AI. Use skill_demand_tool? Not listed; we have only search_jobs and tavily_search. Use tavily_search for demand info. Then search_jobs for job openings with skill "generative AI" in India.\n\nLet\'s query demand: "generative AI industry demand 2024 India salary trends".', 'tool_calls': [{'id': 'fc_f4d635b5-506e-4b2a-9b80-b65186c1861e', 'function': {'arguments': '{"query":"generative AI industry demand 2024 India salary trends","search_depth":"advanced","time_range":"year"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tok

In [158]:
print(response["messages"][-1].content)

**Generative AI – Industry Demand Snapshot (India, 2024‑2025)**  

1. **Market Size & Growth**  
   • The Indian generative‑AI market is projected to expand from **≈ ₹85.3 billion in 2024 to ₹671.8 billion by 2030**, a **compound annual growth rate (CAGR) of ~42 %**. (ResearchAndMarkets, 2024)  
   • Overall AI market in India was valued at **$8.6 billion (≈ ₹720 billion) in 2024** and is expected to reach **$54 billion by 2035** (CAGR ≈ 18 %). Generative AI accounts for the fastest‑growing segment within this broader AI landscape.  

2. **Talent Demand**  
   • A 2024 Deloitte “State of Generative AI in Enterprises” survey shows **27 % of Indian IT firms already use generative‑AI tools**, and **over 50 % plan to double AI‑related hiring in the next 12‑18 months**.  
   • The “State of the Developer Nation” report (2024) lists **5.8 million software developers in India**, with a sharp increase in demand for skills such as prompt engineering, LLM fine‑tuning, and AI‑product development.